# 04 Save, publish, and share

Run this in the **same Colab account** that has the trained checkpoint on Drive. CPU is enough.

It will:

1. Export INT8 ONNX from `final_model`
2. Copy the ONNX onto Drive
3. Upload weights to Hugging Face (`Saalil/Assesment_SR-model`)
4. Deploy the Gradio Space (`Saalil/Assesment_SR`)
5. Commit the small ONNX + metrics to GitHub
6. Print a public Gradio link you can share immediately

**Do not paste tokens into the notebook.** Add Colab secrets named `HF_TOKEN` and `GITHUB_TOKEN`, or type them into the hidden prompt.

- GitHub token: `repo` scope at https://github.com/settings/tokens
- HF token: **write** at https://huggingface.co/settings/tokens

Checkpoint this notebook looks for:

`/content/drive/MyDrive/smart_turn_runs/partial_unfreeze/final_model`

In [ ]:
from __future__ import annotations

import json
import os
import shutil
import subprocess
import sys
import tempfile
from pathlib import Path

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

REPO_URL = "https://github.com/Saaalil/ShipRocket-assesment.git"
REPO = Path("/content/ShipRocket-assesment" if IN_COLAB else ".")
DRIVE_EXPORT = Path("/content/drive/MyDrive/smart_turn_runs/partial_unfreeze")
DRIVE_CKPT = DRIVE_EXPORT / "final_model"
GITHUB_REPO = "Saaalil/ShipRocket-assesment"
HF_MODEL = "Saalil/Assesment_SR-model"
HF_SPACE = "Saalil/Assesment_SR"

if IN_COLAB:
    from google.colab import drive

    drive.mount("/content/drive")
    if REPO.exists():
        pulled = subprocess.call(["git", "-C", str(REPO), "pull", "--ff-only"])
        if pulled != 0:
            print("git pull skipped (local edits). Using this clone.")
    else:
        subprocess.check_call(["git", "clone", REPO_URL, str(REPO)])
    os.chdir(REPO)
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-e", ".[train,demo]"])
else:
    os.chdir(REPO)

print("cwd", Path.cwd())

In [ ]:
def is_checkpoint(path: Path) -> bool:
    if not path.is_dir() or not (path / "config.json").exists():
        return False
    return (path / "model.safetensors").exists() or any(path.glob("*.safetensors"))


def find_checkpoint() -> Path:
    env = os.environ.get("SMART_TURN_CHECKPOINT", "").strip()
    candidates = [Path(env)] if env else []
    candidates.extend([DRIVE_CKPT, Path("artifacts/final_model")])
    shortcut = Path("/content/drive/.shortcut-targets-by-id")
    if shortcut.exists():
        for path in shortcut.rglob("final_model"):
            if path.is_dir():
                candidates.append(path)
    for path in candidates:
        try:
            if path.exists() and is_checkpoint(path):
                return path
        except OSError:
            continue
    raise FileNotFoundError(
        "Could not find final_model. Mount Drive or set SMART_TURN_CHECKPOINT."
    )


ckpt = find_checkpoint()
print("checkpoint:", ckpt)
print("files:", sorted(p.name for p in ckpt.iterdir())[:24])
Path("artifacts").mkdir(exist_ok=True)
subprocess.check_call(
    [
        sys.executable,
        "scripts/export_onnx.py",
        "--config",
        "configs/partial_unfreeze.yaml",
        "--checkpoint",
        str(ckpt),
    ]
)
onnx = Path("artifacts/model_int8.onnx")
fp32 = Path("artifacts/model_fp32.onnx")
if not onnx.exists():
    onnx = fp32
if not onnx.exists():
    raise FileNotFoundError("Export finished but no ONNX file was written.")
if IN_COLAB:
    DRIVE_EXPORT.mkdir(parents=True, exist_ok=True)
    for src in [onnx, fp32, Path("artifacts/model_int8.json"), Path("artifacts/model_fp32.json")]:
        if src.exists():
            shutil.copy2(src, DRIVE_EXPORT / src.name)
            print("drive copy", DRIVE_EXPORT / src.name)
print("serving file", onnx, f"({onnx.stat().st_size / 1024 / 1024:.1f} MB)")

In [ ]:
EVAL = {
    "run": "partial_unfreeze",
    "checkpoint": str(ckpt),
    "onnx": str(onnx),
    "eval_count": 8422,
    "epochs": 2,
    "steps": 1444,
    "threshold": 0.5,
    "final": {
        "eval_loss": 0.6493,
        "eval_accuracy": 0.6178,
        "eval_macro_f1": 0.6174,
        "eval_complete_f1": 0.6288,
        "eval_incomplete_f1": 0.606,
        "eval_roc_auc": 0.6697,
        "eval_tp": 2727,
        "eval_tn": 2476,
        "eval_fp": 1794,
        "eval_fn": 1425,
    },
}
metrics_path = Path("reports/partial_unfreeze_eval.json")
metrics_path.parent.mkdir(parents=True, exist_ok=True)
if metrics_path.exists():
    saved = json.loads(metrics_path.read_text(encoding="utf-8"))
    saved["checkpoint"] = str(ckpt)
    saved["onnx"] = str(onnx)
    metrics_path.write_text(json.dumps(saved, indent=2), encoding="utf-8")
    EVAL = saved
else:
    metrics_path.write_text(json.dumps(EVAL, indent=2), encoding="utf-8")

final = EVAL.get("final", EVAL)
print("validation (n=8422, threshold=0.5)")
print(f"  accuracy     {final['eval_accuracy']}")
print(f"  macro F1     {final['eval_macro_f1']}")
print(f"  complete F1  {final['eval_complete_f1']}")
print(f"  incomplete F1 {final['eval_incomplete_f1']}")
print(f"  ROC-AUC      {final['eval_roc_auc']}")

try:
    subprocess.check_call([sys.executable, "scripts/benchmark_cpu.py", "--onnx", str(onnx), "--repeats", "20"])
except subprocess.CalledProcessError as exc:
    print("benchmark skipped:", exc)

## Tokens

Preferred: Colab left sidebar → key icon → secrets `HF_TOKEN` and `GITHUB_TOKEN`.

Otherwise the next cell asks for them without echoing.

In [ ]:
from getpass import getpass

from huggingface_hub import HfApi, login


def read_secret(name: str) -> str:
    if os.environ.get(name):
        return os.environ[name].strip()
    if IN_COLAB:
        try:
            from google.colab import userdata

            value = userdata.get(name)
            if value:
                return str(value).strip()
        except Exception:
            pass
    return getpass(f"{name} (write token, hidden): ").strip()


HF_TOKEN = read_secret("HF_TOKEN")
GITHUB_TOKEN = read_secret("GITHUB_TOKEN")
if not HF_TOKEN or not GITHUB_TOKEN:
    raise ValueError("Both HF_TOKEN and GITHUB_TOKEN are required.")
os.environ["HF_TOKEN"] = HF_TOKEN
login(token=HF_TOKEN, add_to_git_credential=False)
api = HfApi(token=HF_TOKEN)
print("Hugging Face login ok")

In [ ]:
def upload_file(repo: str, repo_type: str, local: Path, dest: str) -> None:
    if not local.exists():
        return
    api.upload_file(
        path_or_fileobj=str(local),
        path_in_repo=dest,
        repo_id=repo,
        repo_type=repo_type,
        token=HF_TOKEN,
    )
    print(f"{repo_type}: {dest}")


api.create_repo(HF_MODEL, repo_type="model", exist_ok=True, private=False, token=HF_TOKEN)
upload_file(HF_MODEL, "model", onnx, "model_int8.onnx" if onnx.name.startswith("model_int8") else onnx.name)
upload_file(HF_MODEL, "model", onnx.with_suffix(".json"), "model_int8.json")
upload_file(HF_MODEL, "model", fp32, "model_fp32.onnx")
upload_file(HF_MODEL, "model", fp32.with_suffix(".json"), "model_fp32.json")
upload_file(HF_MODEL, "model", Path("model_card/README.md"), "README.md")
upload_file(HF_MODEL, "model", Path("reports/partial_unfreeze_eval.json"), "metrics.json")
if is_checkpoint(ckpt):
    api.upload_folder(
        folder_path=str(ckpt),
        repo_id=HF_MODEL,
        repo_type="model",
        path_in_repo="pytorch",
        token=HF_TOKEN,
        allow_patterns=["*.safetensors", "*.json", "*.txt", "config.json", "preprocessor_config.json"],
    )
    print("model: pytorch/")
print("weights", f"https://huggingface.co/{HF_MODEL}")

spaces_readme = Path("spaces/README.md")
if not spaces_readme.exists():
    spaces_readme.parent.mkdir(parents=True, exist_ok=True)
    spaces_readme.write_text(
        "---\n"
        "title: Assesment SR\n"
        "emoji: \U0001f3a4\n"
        "colorFrom: indigo\n"
        "colorTo: blue\n"
        "sdk: gradio\n"
        "sdk_version: 6.25.0\n"
        "app_file: app.py\n"
        "pinned: false\n"
        "license: apache-2.0\n"
        "---\n\n"
        "# Shiprocket turn detection\n",
        encoding="utf-8",
    )
req = Path("spaces/requirements.txt")
if not req.exists():
    req.write_text(
        "gradio==6.25.0\nonnxruntime>=1.18\ntransformers>=4.44\n"
        "numpy>=1.26\nsoundfile>=0.12\nhuggingface_hub>=0.25.2\npyyaml>=6.0\n",
        encoding="utf-8",
    )
pkg = Path("spaces/packages.txt")
if not pkg.exists():
    pkg.write_text("libsndfile1\nffmpeg\n", encoding="utf-8")
Path("demo").mkdir(exist_ok=True)
if not Path("demo/__init__.py").exists():
    Path("demo/__init__.py").write_text("", encoding="utf-8")

api.create_repo(HF_SPACE, repo_type="space", space_sdk="gradio", exist_ok=True, private=False, token=HF_TOKEN)
stage = Path(tempfile.mkdtemp(prefix="hf-space-"))
space_files = [
    (Path("app.py"), "app.py"),
    (Path("demo/__init__.py"), "demo/__init__.py"),
    (Path("demo/gradio_app.py"), "demo/gradio_app.py"),
    (spaces_readme, "README.md"),
    (req, "requirements.txt"),
    (pkg, "packages.txt"),
    (onnx, "artifacts/model_int8.onnx"),
    (onnx.with_suffix(".json"), "artifacts/model_int8.json"),
]
for local, dest in space_files:
    if not local.exists():
        continue
    target = stage / dest
    target.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(local, target)
src_dir = Path("src/smart_turn")
if src_dir.exists():
    for path in src_dir.rglob("*.py"):
        target = stage / path.as_posix()
        target.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(path, target)
(stage / "app.py").write_text(
    "from __future__ import annotations\n\n"
    "import os\n"
    "import sys\n"
    "from pathlib import Path\n\n"
    "_ROOT = Path(__file__).resolve().parent\n"
    "sys.path[:0] = [str(_ROOT), str(_ROOT / \"src\")]\n\n"
    "from demo.gradio_app import build_demo\n\n"
    "demo = build_demo()\n\n"
    "if __name__ == \"__main__\":\n"
    "    demo.launch(share=os.environ.get(\"GRADIO_SHARE\") == \"1\")\n",
    encoding="utf-8",
)
api.upload_folder(folder_path=str(stage), repo_id=HF_SPACE, repo_type="space", token=HF_TOKEN)
print("demo", f"https://huggingface.co/spaces/{HF_SPACE}")

In [ ]:
def run_git(args: list[str], env: dict | None = None, check: bool = True) -> subprocess.CompletedProcess:
    result = subprocess.run(
        ["git", *args],
        env=env,
        check=False,
        capture_output=True,
        text=True,
    )
    if check and result.returncode != 0:
        blob = f"{result.stdout}\n{result.stderr}"
        for secret in (GITHUB_TOKEN, os.environ.get("HF_TOKEN", "")):
            if secret:
                blob = blob.replace(secret, "***")
        raise RuntimeError(f"git {' '.join(args)} failed ({result.returncode}):\n{blob.strip()}")
    return result


def git_push() -> None:
    askpass = Path("/tmp/colab-git-askpass.sh")
    askpass.write_text(
        "#!/bin/sh\n"
        'case "$1" in\n'
        "  *[Uu]sername*) echo x-access-token ;;\n"
        '  *) echo "$GIT_PASSWORD" ;;\n'
        "esac\n",
        encoding="utf-8",
    )
    askpass.chmod(0o700)
    env = os.environ.copy()
    env.update(
        {
            "GIT_ASKPASS": str(askpass),
            "GIT_PASSWORD": GITHUB_TOKEN,
            "GIT_TERMINAL_PROMPT": "0",
            "GCM_INTERACTIVE": "never",
        }
    )
    run_git(["-c", "credential.helper=", "push", "origin", "HEAD:main"], env=env)


GIT_ID = ["-c", "user.name=Saalil", "-c", "user.email=Saaalil@users.noreply.github.com"]
run_git(["remote", "set-url", "origin", f"https://github.com/{GITHUB_REPO}.git"])

nb_src = Path("/content/04_save_publish_demo_colab.ipynb")
nb_dst = Path("notebooks/04_save_publish_demo_colab.ipynb")
if nb_src.exists() and not nb_dst.exists():
    nb_dst.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(nb_src, nb_dst)

add = [
    "reports/partial_unfreeze_eval.json",
    "model_card/README.md",
    "notebooks/04_save_publish_demo_colab.ipynb",
    "spaces/README.md",
    "spaces/requirements.txt",
    "spaces/packages.txt",
    "app.py",
    "demo/gradio_app.py",
    "demo/__init__.py",
    "scripts/publish_space.py",
    "scripts/publish_model.py",
    "src/smart_turn/inference.py",
    "src/smart_turn/constants.py",
]
for path in add:
    if Path(path).exists():
        run_git(["add", path])
if onnx.exists() and onnx.stat().st_size < 95 * 1024 * 1024:
    run_git(["add", "-f", str(onnx)])
    sidecar = onnx.with_suffix(".json")
    if sidecar.exists():
        run_git(["add", "-f", str(sidecar)])
else:
    print("skip GitHub ONNX (missing or >= 95 MB); HF already has the weights")

status = run_git(["status", "--porcelain"], check=False).stdout
if status.strip():
    run_git([*GIT_ID, "commit", "-m", "Publish partial-unfreeze ONNX and validation metrics"])

run_git(["fetch", "origin"], check=False)
ahead = run_git(["rev-list", "--count", "origin/main..HEAD"], check=False)
n_ahead = int((ahead.stdout or "0").strip() or "0") if ahead.returncode == 0 else 1
if n_ahead > 0:
    git_push()
    print("pushed", f"https://github.com/{GITHUB_REPO}")
else:
    print("GitHub already up to date")

In [ ]:
import os
import sys
from pathlib import Path

repo = Path("/content/ShipRocket-assesment")
if repo.exists():
    os.chdir(repo)
for extra in (Path.cwd(), Path.cwd() / "src"):
    if extra.exists() and str(extra) not in sys.path:
        sys.path.insert(0, str(extra))

try:
    import smart_turn  # noqa: F401
except ModuleNotFoundError:
    import subprocess

    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-e", ".[demo]"])

candidates = []
if "onnx" in dir():
    candidates.append(Path(str(onnx)))
candidates.extend(
    [
        Path("artifacts/model_int8.onnx"),
        Path("/content/drive/MyDrive/smart_turn_runs/partial_unfreeze/model_int8.onnx"),
    ]
)
onnx_path = next((p.resolve() for p in candidates if p.exists()), None)
if onnx_path is None:
    raise FileNotFoundError("No ONNX file found. Re-run the export cell first.")
os.environ["SMART_TURN_ONNX_PATH"] = str(onnx_path)

from demo.gradio_app import build_demo

demo = build_demo()
print("Space (permanent):", f"https://huggingface.co/spaces/{HF_SPACE if 'HF_SPACE' in dir() else 'Saalil/Assesment_SR'}")
print("Model repo:", f"https://huggingface.co/{HF_MODEL if 'HF_MODEL' in dir() else 'Saalil/Assesment_SR-model'}")
print("GitHub:", f"https://github.com/{GITHUB_REPO if 'GITHUB_REPO' in dir() else 'Saaalil/ShipRocket-assesment'}")
print("The next line starts a temporary public Gradio URL for this runtime.")
demo.launch(share=True, debug=False)

## Share these

- Demo (this is the link to send): https://huggingface.co/spaces/Saalil/Assesment_SR
- Weights: https://huggingface.co/Saalil/Assesment_SR-model
- Code: https://github.com/Saaalil/ShipRocket-assesment

The last code cell also prints a `*.gradio.live` URL. That one dies when the Colab runtime stops. The Space URL stays.

If the Space build is still queued, wait a couple of minutes and refresh.